# 06. bisect

**Tier:** Intermediate. **Why you'd use this:** You keep a sorted list of high scores and need to insert a new score in the right place, or ask how many scores beat a threshold, without scanning from the left each time.

**After this lesson you can:** find an insertion point with `bisect_left`; insert with `insort`; count values below a limit.
About 25 minutes. **Uses:** [Sorting](../07-iteration/03-sorting-lesson.ipynb), [Lists](../04-collections/01-lists-lesson.ipynb)


## Quick review

1. Why must `cache` arguments be hashable?
2. What does `islice` do to a lazy stream?

<details><summary>Answers</summary>

They become dict keys inside the cache. It pulls a finite window and stops.
</details>


## The idea

`bisect` binary-searches a sorted list. `bisect_left` returns the index where a value should land to keep order. `insort` inserts there. The list must already be sorted. The search is logarithmic; the insert still shifts tail elements, so inserts are not free.

<details><summary>Explain more</summary>

Use `bisect_left` when equal values should land at the left edge. Use `bisect_right` when they should land after existing equals. Counting values below a limit is just the left insertion index of that limit.
</details>


## See it work

Keep a sorted scoreboard and count scores below a cut-off.


In [1]:
import bisect


def add_score(scores, value):
    # 1. Insert into an already sorted list.
    ordered = list(scores)
    bisect.insort(ordered, value)
    return ordered


def count_below(scores, limit):
    # 2. The left insertion point is the count of smaller values.
    return bisect.bisect_left(scores, limit)


print(add_score([10, 20, 30, 40], 25))
print(count_below([10, 20, 30, 40], 25))

[10, 20, 25, 30, 40]
2


## Check yourself

Answer these before you run the cell.

1. What index does `bisect_left([10, 20, 30], 20)` return?
2. Does `insort` require the list to be sorted already?


In [2]:
print(bisect.bisect_left([10, 20, 30], 20))
print(bisect.bisect_right([10, 20, 30], 20))
scores = [10, 20, 30]
bisect.insort(scores, 20)
print(scores)

1
2
[10, 20, 20, 30]


<details><summary>Answers</summary>

`bisect_left` returns `1`. Yes. `insort` trusts the existing order and only finds the landing index.
</details>


## Why it works this way

Binary search compares the target to the middle value and discards half the remaining range each step. That is O(log n) comparisons on a random-access sequence. A list gives random access, so `bisect` fits. A structure that only walks links would not give the same cost. Deep dive: [How list works](../../internals/03-lists/01-dynamic-array-lesson.ipynb).

`insort` still has to move the tail of the list to open a slot, which is O(n). Binary search finds the index quickly; the insert pays the list's shift cost. For heavy insert workloads a tree or blist-style structure wins. For occasional inserts into a modest sorted list, or for many rank queries between inserts, `bisect` is the right tool. The search cost and the mutation cost are different bills.

`bisect_left` and `bisect_right` disagree only on ties. Left means "before existing equals." Right means "after existing equals." Counting values strictly below `limit` is `bisect_left(scores, limit)` because every index before that point holds a smaller value. Counting values less than or equal to `limit` uses `bisect_right`.

The design is a tiny toolkit over sorted lists, not a new container. You keep the list. `bisect` only answers where and helps insert. Sorting first remains your job when the input arrives unsorted. Calling `insort` on disorder does not repair the list; it only trusts the binary-search assumption and leaves you with a wrong sequence that looks almost sorted. Sort first when the data arrives unordered, then use `bisect` for ranks and inserts.


In [3]:
scores = [10, 20, 20, 40]
print(bisect.bisect_left(scores, 20))
print(bisect.bisect_right(scores, 20))
print(bisect.bisect_left(scores, 25))
ordered = list(scores)
bisect.insort(ordered, 25)
print(ordered)

1
3
3
[10, 20, 20, 25, 40]


## When to use it

Use `bisect` on a list you already keep sorted. Use it to find ranks and insertion points. Use a heap when you only need repeated minima, not full order. Don't call `insort` on an unsorted list and trust the result.


## Common mistake

Calling `insort` on an unsorted list. The insert lands by binary search as if the order were valid, so the result is still wrong.


In [4]:
scores = [30, 10, 20]
bisect.insort(scores, 15)
print(scores)

[30, 10, 15, 20]


Sort first, or build the list with `insort` from the start. Binary search only works on ordered data.


In [5]:
scores = []
for value in (30, 10, 20, 15):
    bisect.insort(scores, value)
print(scores)

[10, 15, 20, 30]


## Try it

Change `count_below` to count values less than or equal to `limit`. Which function replaces `bisect_left`, and why?


## Recap

- `bisect_left` finds the insertion index in a sorted list.
- `insort` inserts there. The shift is still O(n).
- Left versus right matters for ties and inclusive counts.

**Next:** [practice](06-bisect-practice.ipynb)
